In [ ]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np

import os
if "original_dir" not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, "../../.."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "checkpoints")

with open("data/traces_dataset/nonograms/test.jsonl", "r") as f:
    json_dataset = [json.loads(line) for line in f.readlines()]
traces = [
    {"puzzle_idx": i, "method": m, **run}
    for i, p in enumerate(json_dataset)
    for m, d in p["traces"].items()
    for run in next(iter(d.values()))
]

df = pd.json_normalize(traces)
df.head()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

TARGET_LEN = 50
RANDOM_STATE = 0


# ---------------------------------------------------------------------------
# 1) Resample each sequence to TARGET_LEN time points (normalized time)
# ---------------------------------------------------------------------------
def resample_sequence(steps, target_len=TARGET_LEN):
    """steps: list/array of 2D grids. Returns a flat (target_len*H*W,) vector."""
    arr = np.asarray(steps, dtype=float)          # (T, H, W)
    if arr.ndim != 3:
        raise ValueError(f"Expected (T, H, W), got {arr.shape}")
    T, H, W = arr.shape
    flat = arr.reshape(T, H * W)                  # (T, H*W)

    if T == 1:                                    # no time axis to interpolate
        return np.tile(flat[0], target_len)

    t_old = np.linspace(0.0, 1.0, T)
    t_new = np.linspace(0.0, 1.0, target_len)
    out = np.empty((target_len, H * W), dtype=float)
    for j in range(H * W):
        out[:, j] = np.interp(t_new, t_old, flat[:, j])
    return out.reshape(-1)


# ---------------------------------------------------------------------------
# 2) Build trivial features Z
# ---------------------------------------------------------------------------
def _mse_stats(losses):
    if losses is None or len(losses) == 0:
        return {}
    a = np.asarray(losses, dtype=float)
    out = {
        "mse_mean":  np.nanmean(a),
        "mse_std":   np.nanstd(a),
        "mse_first": a[0],
        "mse_last":  a[-1],
        "mse_min":   np.nanmin(a),
        "mse_max":   np.nanmax(a),
    }
    valid = ~np.isnan(a)
    if valid.sum() >= 2:
        t = np.linspace(0, 1, len(a))[valid]
        out["mse_slope"] = np.polyfit(t, a[valid], 1)[0]
    else:
        out["mse_slope"] = 0.0
    return out


def build_trivial_features(df):
    n_steps = df["steps"].apply(len).astype(float).values

    grid_mean, grid_std = [], []
    first_mean, last_mean = [], []
    first_std, last_std = [], []
    for s in df["steps"]:
        arr = np.asarray(s, dtype=float)
        grid_mean.append(arr.mean())
        grid_std.append(arr.std())
        first_mean.append(arr[0].mean())
        last_mean.append(arr[-1].mean())
        first_std.append(arr[0].std())
        last_std.append(arr[-1].std())

    feats = pd.DataFrame({
        "n_steps":     n_steps,
        "log_n_steps": np.log(n_steps),
        "grid_mean":   grid_mean,
        "grid_std":    grid_std,
        "first_mean":  first_mean,
        "last_mean":   last_mean,
        "first_std":   first_std,
        "last_std":    last_std,
        "delta_mean":  np.array(last_mean) - np.array(first_mean),
    }, index=df.index)

    mse_df = df["mse_losses"].apply(_mse_stats).apply(pd.Series)
    feats = pd.concat([feats, mse_df], axis=1)

    stop = pd.get_dummies(df["stopping_reason"].astype(str),
                          prefix="stop").astype(float)
    feats = pd.concat([feats, stop], axis=1)

    return feats


# ---------------------------------------------------------------------------
# 3) Full pipeline: resample -> residualize against Z -> PCA
# ---------------------------------------------------------------------------
def fit_residualized_pca(df, n_components=20, target_len=TARGET_LEN,
                         extra_trivial_cols=None, verbose=True):
    # (a) Resample every sequence to a fixed length
    X = np.stack([resample_sequence(s, target_len) for s in df["steps"].values])

    # (b) Trivial features
    Z_df = build_trivial_features(df)
    if extra_trivial_cols:
        Z_df = pd.concat([Z_df, df[extra_trivial_cols].astype(float)], axis=1)

    # (c) Impute + standardize
    X = SimpleImputer(strategy="mean").fit_transform(X)
    Z = SimpleImputer(strategy="mean").fit_transform(Z_df.values)
    Z = StandardScaler().fit_transform(Z)

    Xs = StandardScaler().fit_transform(X)

    # (d) Residualize X against Z:  X_resid = X - Z @ B
    reg = LinearRegression().fit(Z, Xs)
    X_resid = Xs - reg.predict(Z)

    # (e) PCA on residuals
    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    scores = pca.fit_transform(X_resid)

    if verbose:
        print(f"Resampled X: {X.shape}   Z: {Z.shape}")
        print("PC diagnostics (max |corr| with any trivial feature):")
        for i in range(scores.shape[1]):
            corrs = [
                abs(np.corrcoef(scores[:, i], Z[:, j])[0, 1])
                if Z[:, j].std() > 0 else 0.0
                for j in range(Z.shape[1])
            ]
            print(f"  PC{i+1:>2}: var={pca.explained_variance_ratio_[i]:.4f}  "
                  f"max|corr|={max(corrs):.3f}")

    return {
        "scores":       scores,
        "pca":          pca,
        "X_resid":      X_resid,
        "X_scaled":     Xs,
        "Z":            Z,
        "Z_columns":    Z_df.columns.tolist(),
        "residualizer": reg,
    }



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

method_dummies = pd.get_dummies(
    df["method"],
    prefix="method",
    dtype=float,
)

df_pca = pd.concat([df, method_dummies], axis=1)

result = fit_residualized_pca(
    df_pca,
    n_components=20,
    target_len=TARGET_LEN,
    extra_trivial_cols=method_dummies.columns.tolist(),
    verbose=True,
)

scores  = result["scores"]        # (n, k)
pca     = result["pca"]
Z_cols  = result["Z_columns"]

meta = df.reset_index(drop=True)  # align with scores row order
def plot_scree(pca, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    evr = pca.explained_variance_ratio_
    ax.bar(range(1, len(evr) + 1), evr, alpha=0.7)
    ax.plot(range(1, len(evr) + 1), np.cumsum(evr),
            marker="o", color="C1", label="cumulative")
    ax.set_xlabel("PC"); ax.set_ylabel("explained variance ratio")
    ax.set_title("Scree (after residualizing on Z)")
    ax.legend()
    return ax

In [ ]:
def plot_pc_scatter(scores, meta, x=0, y=1, color_by="method",
                    style_by=None, figsize=(6, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    if style_by is None:
        for grp, sub in meta.groupby(color_by):
            idx = sub.index.values
            ax.scatter(scores[idx, x], scores[idx, y],
                       s=30, alpha=0.7, label=str(grp))
    else:
        for (g1, g2), sub in meta.groupby([color_by, style_by]):
            idx = sub.index.values
            ax.scatter(scores[idx, x], scores[idx, y],
                       s=30, alpha=0.7, label=f"{g1} / {g2}")
    ax.set_xlabel(f"PC{x+1} ({pca.explained_variance_ratio_[x]:.1%})")
    ax.set_ylabel(f"PC{y+1} ({pca.explained_variance_ratio_[y]:.1%})")
    ax.axhline(0, lw=0.5, color="k", alpha=0.3)
    ax.axvline(0, lw=0.5, color="k", alpha=0.3)
    ax.legend(fontsize=8, loc="best", frameon=False)
    return fig

plot_pc_scatter(scores, meta, x=0, y=1, color_by="method")
plot_pc_scatter(scores, meta, x=0, y=1, color_by="stopping_reason")
plot_pc_scatter(scores, meta, x=0, y=1, color_by="method",
                style_by="stopping_reason")

In [ ]:
# (c) Is it noise on a sphere? Check residual rank & isotropy.
from scipy.stats import chi2
R = result["X_resid"]
U, S, Vt = np.linalg.svd(R - R.mean(0), full_matrices=False)
evr = S**2 / (S**2).sum()
print("top-10 evr:", evr[:10])
print("participation ratio:", 1.0 / (evr**2).sum())   # effective dim

In [ ]:
from itertools import combinations

def pairwise_separation(X, y):
    rows = []
    for a, b in combinations(np.unique(y), 2):
        Xa, Xb = X[y==a], X[y==b]
        mu = Xa.mean(0) - Xb.mean(0)
        # pooled within-class covariance
        Sa = np.cov(Xa.T); Sb = np.cov(Xb.T)
        Sp = ((len(Xa)-1)*Sa + (len(Xb)-1)*Sb) / (len(Xa)+len(Xb)-2)
        d_maha = float(np.sqrt(mu @ np.linalg.pinv(Sp) @ mu))
        # simple centroid distance for reference
        rows.append((a, b, np.linalg.norm(mu), d_maha))
    return pd.DataFrame(rows, columns=["method_A","method_B",
                                       "centroid_dist","mahalanobis"])
print(pairwise_separation(X, y).round(3))

In [ ]:
# =============================================================================
# Full pipeline: resample -> residualize trivial features -> PCA
#                + separability diagnostics
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             silhouette_score)
from sklearn.model_selection import (LeaveOneOut, StratifiedKFold,
                                     cross_val_predict, cross_val_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

TARGET_LEN = 50
RANDOM_STATE = 0
N_PCS = 20

# -----------------------------------------------------------------------------
# 1) Resample each sequence to TARGET_LEN time points (normalized time)
# -----------------------------------------------------------------------------
def resample_sequence(steps, target_len=TARGET_LEN):
    arr = np.asarray(steps, dtype=float)
    if arr.ndim != 3:
        raise ValueError(f"Expected (T, H, W), got {arr.shape}")
    T, H, W = arr.shape
    flat = arr.reshape(T, H * W)
    if T == 1:
        return np.tile(flat[0], target_len)
    t_old = np.linspace(0.0, 1.0, T)
    t_new = np.linspace(0.0, 1.0, target_len)
    out = np.empty((target_len, H * W), dtype=float)
    for j in range(H * W):
        out[:, j] = np.interp(t_new, t_old, flat[:, j])
    return out.reshape(-1)


# -----------------------------------------------------------------------------
# 2) Trivial feature builder (Z)
# -----------------------------------------------------------------------------
def _mse_stats(losses):
    if losses is None or len(losses) == 0:
        return {k: 0.0 for k in
                ["mse_mean","mse_std","mse_first","mse_last",
                 "mse_min","mse_max","mse_slope"]}
    a = np.asarray(losses, dtype=float)
    out = {
        "mse_mean":  np.nanmean(a),
        "mse_std":   np.nanstd(a),
        "mse_first": a[0],
        "mse_last":  a[-1],
        "mse_min":   np.nanmin(a),
        "mse_max":   np.nanmax(a),
    }
    valid = ~np.isnan(a)
    if valid.sum() >= 2:
        t = np.linspace(0, 1, len(a))[valid]
        out["mse_slope"] = np.polyfit(t, a[valid], 1)[0]
    else:
        out["mse_slope"] = 0.0
    return out


def build_trivial_features(df, drop_method_identifying=False):
    n_steps = df["steps"].apply(len).astype(float).values

    grid_mean, grid_std = [], []
    first_mean, last_mean = [], []
    first_std, last_std = [], []
    for s in df["steps"]:
        arr = np.asarray(s, dtype=float)
        grid_mean.append(arr.mean())
        grid_std.append(arr.std())
        first_mean.append(arr[0].mean())
        last_mean.append(arr[-1].mean())
        first_std.append(arr[0].std())
        last_std.append(arr[-1].std())

    feats = pd.DataFrame({
        "n_steps":     n_steps,
        "log_n_steps": np.log(n_steps),
        "grid_mean":   grid_mean,
        "grid_std":    grid_std,
        "first_mean":  first_mean,
        "last_mean":   last_mean,
        "first_std":   first_std,
        "last_std":    last_std,
        "delta_mean":  np.array(last_mean) - np.array(first_mean),
    }, index=df.index)

    mse_df = df["mse_losses"].apply(_mse_stats).apply(pd.Series)
    feats = pd.concat([feats, mse_df], axis=1)

    # stopping_reason is often highly correlated with method.
    if not drop_method_identifying:
        stop = pd.get_dummies(df["stopping_reason"].astype(str),
                              prefix="stop").astype(float)
        feats = pd.concat([feats, stop], axis=1)

    return feats


# -----------------------------------------------------------------------------
# 3) Full pipeline: resample -> residualize against Z -> PCA
# -----------------------------------------------------------------------------
def fit_residualized_pca(df, n_components=N_PCS, target_len=TARGET_LEN,
                         extra_trivial_cols=None,
                         drop_method_identifying=False,
                         verbose=True):
    X = np.stack([resample_sequence(s, target_len) for s in df["steps"].values])
    Z_df = build_trivial_features(df,
                                  drop_method_identifying=drop_method_identifying)
    if extra_trivial_cols:
        Z_df = pd.concat([Z_df, df[extra_trivial_cols].astype(float)], axis=1)

    X = SimpleImputer(strategy="mean").fit_transform(X)
    Z = SimpleImputer(strategy="mean").fit_transform(Z_df.values)
    Z = StandardScaler().fit_transform(Z)
    Xs = StandardScaler().fit_transform(X)

    reg = LinearRegression().fit(Z, Xs)
    X_resid = Xs - reg.predict(Z)

    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    scores = pca.fit_transform(X_resid)

    if verbose:
        print(f"Resampled X: {X.shape}   Z: {Z.shape}")
        print("PC diagnostics (max |corr| with any trivial feature):")
        for i in range(scores.shape[1]):
            corrs = [abs(np.corrcoef(scores[:, i], Z[:, j])[0, 1])
                     if Z[:, j].std() > 0 else 0.0
                     for j in range(Z.shape[1])]
            print(f"  PC{i+1:>2}: var={pca.explained_variance_ratio_[i]:.4f}  "
                  f"max|corr|={max(corrs):.3f}")

    return {
        "scores":       scores,
        "pca":          pca,
        "X_resid":      X_resid,
        "X_scaled":     Xs,
        "Z":            Z,
        "Z_columns":    Z_df.columns.tolist(),
        "residualizer": reg,
    }


# -----------------------------------------------------------------------------
# 4) Separability diagnostics
# -----------------------------------------------------------------------------
def pairwise_separation(X, y):
    rows = []
    for a, b in combinations(np.unique(y), 2):
        Xa, Xb = X[y == a], X[y == b]
        if len(Xa) < 2 or len(Xb) < 2:
            continue
        mu = Xa.mean(0) - Xb.mean(0)
        Sa = np.cov(Xa.T); Sb = np.cov(Xb.T)
        Sp = ((len(Xa)-1)*Sa + (len(Xb)-1)*Sb) / (len(Xa)+len(Xb)-2)
        d_maha = float(np.sqrt(mu @ np.linalg.pinv(Sp) @ mu))
        rows.append((a, b, np.linalg.norm(mu), d_maha))
    return pd.DataFrame(rows, columns=["method_A","method_B",
                                       "centroid_dist","mahalanobis"])


# --- Fix 1: robust y ---
def run_separability_report(result, meta, label_col="method",
                            n_pcs_use=10, cv_folds=5):
    meta = meta.reset_index(drop=True)
    X = np.asarray(result["scores"][:, :n_pcs_use])
    y = np.asarray(meta[label_col])          # <- fix
    chance = pd.Series(y).value_counts(normalize=True).max()

    print("\n" + "="*70)
    print(f"SEPARABILITY REPORT  (label = {label_col})")
    print("="*70)
    print(f"n = {len(y)}, n_classes = {len(np.unique(y))}, chance = {chance:.3f}")

    # --- class counts guard ---
    counts = pd.Series(y).value_counts()
    min_count = counts.min()
    print("class counts:\n", counts.to_string())

    # [0] Silhouette
    print("\n[0] Silhouette scores (raw / resid / PCs)")
    try:
        print(f"    raw = {silhouette_score(result['X_scaled'], y):+.3f}   "
              f"resid = {silhouette_score(result['X_resid'], y):+.3f}   "
              f"PCs = {silhouette_score(X, y):+.3f}")
    except ValueError as e:
        print(f"    (skipped: {e})")

    # CV splitter: guarantee each class appears in train folds
    if min_count < cv_folds:
        cv = LeaveOneOut() if len(y) < 100 else StratifiedKFold(
            n_splits=max(2, min_count), shuffle=True,
            random_state=RANDOM_STATE)
    else:
        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True,
                             random_state=RANDOM_STATE)

    # [1] kNN
    print("\n[1] kNN (k=5) CV accuracy")
    try:
        acc = cross_val_score(KNeighborsClassifier(5), X, y, cv=cv).mean()
        print(f"    acc = {acc:.3f}   chance = {chance:.3f}   "
              f"lift = {acc - chance:+.3f}")
    except Exception as e:
        print(f"    (skipped: {e})")

    # [2] LDA
    print("\n[2] LDA CV accuracy")
    try:
        acc = cross_val_score(LinearDiscriminantAnalysis(), X, y,
                              cv=cv).mean()
        print(f"    acc = {acc:.3f}   chance = {chance:.3f}   "
              f"lift = {acc - chance:+.3f}")
    except Exception as e:
        print(f"    (skipped: {e})")

    # [3] LogReg  (no multi_class kwarg)
    print("\n[3] Logistic regression CV accuracy")
    try:
        clf = LogisticRegression(max_iter=5000)
        acc = cross_val_score(clf, X, y, cv=cv).mean()
        print(f"    acc = {acc:.3f}   chance = {chance:.3f}   "
              f"lift = {acc - chance:+.3f}")
    except Exception as e:
        print(f"    (skipped: {e})")

    # [4] RF
    print("\n[4] Random forest CV accuracy")
    try:
        clf = RandomForestClassifier(n_estimators=400,
                                     random_state=RANDOM_STATE, n_jobs=-1)
        acc = cross_val_score(clf, X, y, cv=cv).mean()
        print(f"    acc = {acc:.3f}   chance = {chance:.3f}   "
              f"lift = {acc - chance:+.3f}")
    except Exception as e:
        print(f"    (skipped: {e})")

    # [5] Pairwise Mahalanobis
    print("\n[5] Pairwise separation (Euclidean / Mahalanobis)")
    sep = pairwise_separation(X, y)
    if len(sep):
        print(sep.round(3).to_string(index=False))

    # [6] PERMANOVA
    print("\n[6] PERMANOVA (999 permutations)")
    try:
        from skbio.stats.distance import permanova, DistanceMatrix
        D = squareform(pdist(X))
        dm = DistanceMatrix(D, ids=[str(i) for i in range(len(y))])
        print(permanova(dm, grouping=y, permutations=999))
    except ImportError:
        print("    (skipped: pip install scikit-bio)")
    except Exception as e:
        print(f"    (skipped: {e})")

    # [7] Confusion matrices
    print("\n[7] Confusion matrices (cross-validated)")
    for name, clf in [("logreg", LogisticRegression(max_iter=5000)),
                      ("rf", RandomForestClassifier(n_estimators=400,
                                                    random_state=RANDOM_STATE,
                                                    n_jobs=-1))]:
        try:
            yhat = cross_val_predict(clf, X, y, cv=cv)
            print(f"\n  --- {name} ---")
            print(classification_report(y, yhat, digits=3, zero_division=0))
        except Exception as e:
            print(f"  (skipped {name}: {e})")

# -----------------------------------------------------------------------------
# 5) Visualizations
# -----------------------------------------------------------------------------
def plot_scree(pca, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    evr = pca.explained_variance_ratio_
    ax.bar(range(1, len(evr) + 1), evr, alpha=0.7)
    ax.plot(range(1, len(evr) + 1), np.cumsum(evr),
            marker="o", color="C1", label="cumulative")
    ax.set_xlabel("PC"); ax.set_ylabel("explained variance ratio")
    ax.set_title("Scree (after residualizing on Z)")
    ax.legend()
    return ax


def plot_pc_scatter(scores, meta, x=0, y=1, color_by="method",
                    style_by=None, ax=None, pca=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    meta = meta.reset_index(drop=True)
    if style_by is None:
        for grp, sub in meta.groupby(color_by):
            idx = sub.index.values
            ax.scatter(scores[idx, x], scores[idx, y],
                       s=30, alpha=0.7, label=str(grp))
    else:
        for (g1, g2), sub in meta.groupby([color_by, style_by]):
            idx = sub.index.values
            ax.scatter(scores[idx, x], scores[idx, y],
                       s=30, alpha=0.7, label=f"{g1} / {g2}")
    xl = f"PC{x+1}"; yl = f"PC{y+1}"
    if pca is not None:
        xl += f" ({pca.explained_variance_ratio_[x]:.1%})"
        yl += f" ({pca.explained_variance_ratio_[y]:.1%})"
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.axhline(0, lw=0.5, color="k", alpha=0.3)
    ax.axvline(0, lw=0.5, color="k", alpha=0.3)
    ax.legend(fontsize=8, loc="best", frameon=False)
    return ax


def plot_loading_heatmap(pca, pc_idx, target_len=TARGET_LEN,
                         H=5, W=5, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 6))
    v = pca.components_[pc_idx].reshape(target_len, H * W)
    vmax = np.abs(v).max()
    im = ax.imshow(v, aspect="auto", cmap="RdBu_r",
                   vmin=-vmax, vmax=vmax,
                   extent=[0, H*W, 1, 0])
    ax.set_xlabel("cell index (0..24)")
    ax.set_ylabel("normalized time (0=start, 1=end)")
    ax.set_title(f"PC{pc_idx+1} loading "
                 f"({pca.explained_variance_ratio_[pc_idx]:.1%})")
    plt.colorbar(im, ax=ax, fraction=0.046)
    return ax


def plot_pc_effect(pca, pc_idx, n_sigma=2, target_len=TARGET_LEN,
                   H=5, W=5):
    mean = pca.mean_
    comp = pca.components_[pc_idx]
    scale = np.sqrt(pca.explained_variance_[pc_idx])
    lo = mean - n_sigma * scale * comp
    hi = mean + n_sigma * scale * comp
    ts = np.linspace(0, target_len - 1, 5).astype(int)
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for col, t in enumerate(ts):
        for row, (data, label) in enumerate(
                [(lo, f"-{n_sigma}σ"), (hi, f"+{n_sigma}σ")]):
            g = data.reshape(target_len, H, W)[t]
            ax = axes[row, col]
            ax.imshow(g, cmap="viridis", vmin=0, vmax=1)
            ax.set_xticks([]); ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(label)
            if row == 0:
                ax.set_title(f"t = {t/(target_len-1):.2f}")
    fig.suptitle(f"PC{pc_idx+1} effect (grids at 5 time slices)")
    return fig


def plot_pc_z_corr(result):
    scores = result["scores"]; Z = result["Z"]; Z_cols = result["Z_columns"]
    corr = np.zeros((scores.shape[1], Z.shape[1]))
    for i in range(scores.shape[1]):
        for j in range(Z.shape[1]):
            if Z[:, j].std() > 0:
                corr[i, j] = np.corrcoef(scores[:, i], Z[:, j])[0, 1]
    fig, ax = plt.subplots(figsize=(max(8, len(Z_cols)*0.35),
                                    max(4, scores.shape[1]*0.25)))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(Z_cols)))
    ax.set_xticklabels(Z_cols, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(scores.shape[1]))
    ax.set_yticklabels([f"PC{i+1}" for i in range(scores.shape[1])],
                       fontsize=8)
    ax.set_title("corr(PC score, trivial feature) — should be ~0")
    plt.colorbar(im, ax=ax, fraction=0.03)
    return fig


def plot_pc_vs_length(scores, meta, n_pcs=3):
    n_steps = meta["steps"].apply(len).values
    fig, axes = plt.subplots(1, n_pcs, figsize=(5*n_pcs, 4))
    if n_pcs == 1:
        axes = [axes]
    for ax, k in zip(axes, range(n_pcs)):
        ax.scatter(n_steps, scores[:, k], s=15, alpha=0.6)
        ax.set_xscale("log")
        ax.set_xlabel("n_steps"); ax.set_ylabel(f"PC{k+1}")
        r = np.corrcoef(np.log(n_steps), scores[:, k])[0, 1]
        ax.set_title(f"PC{k+1} vs length (r={r:.2f})")
    plt.tight_layout()
    return fig


def plot_lda_with_ellipses(result, meta, label_col="method",
                           n_pcs_use=10):
    from matplotlib.patches import Ellipse
    X = result["scores"][:, :n_pcs_use]
    y = meta[label_col].values
    lda = LinearDiscriminantAnalysis(n_components=2)
    lda_scores = lda.fit_transform(X, y)

    fig, ax = plt.subplots(figsize=(7, 6))
    for m in np.unique(y):
        s = lda_scores[y == m]
        ax.scatter(s[:, 0], s[:, 1], s=30, alpha=0.6, label=str(m))
        if len(s) >= 3:
            mu = s.mean(0); cov = np.cov(s.T)
            vals, vecs = np.linalg.eigh(cov)
            order = vals.argsort()[::-1]
            vals, vecs = vals[order], vecs[:, order]
            angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
            w, h = 2 * np.sqrt(np.maximum(vals, 0))
            ax.add_patch(Ellipse(mu, w, h, angle=angle,
                                 fill=False, lw=2, alpha=0.8))
    ax.set_xlabel("LD1"); ax.set_ylabel("LD2")
    ax.set_title(f"LDA — maximum linear separability ({label_col})")
    ax.legend(fontsize=8)
    plt.tight_layout()
    return fig, lda_scores


# -----------------------------------------------------------------------------
# 6) One-shot: run everything
# -----------------------------------------------------------------------------
def run_full_analysis(df, label_col="method",
                      drop_method_identifying=False,
                      extra_trivial_cols=None,
                      n_components=N_PCS, target_len=TARGET_LEN):
    """
    df must have columns: puzzle_idx, method, steps, stopping_reason, mse_losses
    Returns dict with result, meta, and figures.
    """
    meta = df.reset_index(drop=True)

    result = fit_residualized_pca(
        meta,
        n_components=n_components,
        target_len=target_len,
        extra_trivial_cols=extra_trivial_cols,
        drop_method_identifying=drop_method_identifying,
        verbose=True,
    )

    report = run_separability_report(result, meta, label_col=label_col)

    figs = {}
    figs["scree"] = plot_scree(result["pca"])
    figs["pc12"] = plt.figure(figsize=(6, 5))
    plot_pc_scatter(result["scores"], meta, 0, 1,
                    color_by=label_col, ax=figs["pc12"].gca(),
                    pca=result["pca"])
    figs["pc13"] = plt.figure(figsize=(6, 5))
    plot_pc_scatter(result["scores"], meta, 0, 2,
                    color_by=label_col, ax=figs["pc13"].gca(),
                    pca=result["pca"])
    figs["loadings"] = plt.figure(figsize=(18, 5))
    for i in range(4):
        plot_loading_heatmap(result["pca"], i, target_len=target_len,
                             ax=figs["loadings"].add_subplot(1, 4, i + 1))
    plt.tight_layout()
    figs["pc_effect_1"] = plot_pc_effect(result["pca"], 0,
                                         target_len=target_len)
    figs["pc_effect_2"] = plot_pc_effect(result["pca"], 1,
                                         target_len=target_len)
    figs["pc_z_corr"] = plot_pc_z_corr(result)
    figs["pc_vs_length"] = plot_pc_vs_length(result["scores"], meta)
    figs["lda"], lda_scores = plot_lda_with_ellipses(result, meta,
                                                     label_col=label_col)

    plt.show()

    return {"result": result, "report": report, "figures": figs,
            "lda_scores": lda_scores}


# =============================================================================
# RUN IT
# =============================================================================
# df = your dataframe (columns: puzzle_idx, method, steps,
#                      stopping_reason, mse_losses)
#
# If methods have different typical lengths / stop reasons and you want
# them NOT removed as trivial:
#   out = run_full_analysis(df, drop_method_identifying=True)
#
# Default (all your listed trivial features removed):
out = run_full_analysis(df, label_col="method")

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold

y = np.asarray(meta["method"])
cv = StratifiedKFold(5, shuffle=True, random_state=0)
chance = pd.Series(y).value_counts(normalize=True).max()

# (a) RAW features, no residualization
X_raw = np.stack([resample_sequence(s) for s in meta["steps"]])
X_raw = StandardScaler().fit_transform(X_raw)
pca_raw = PCA(n_components=100, random_state=0).fit_transform(X_raw)
acc_raw = cross_val_score(
    RandomForestClassifier(400, n_jobs=-1, random_state=0),
    pca_raw, y, cv=cv).mean()
print(f"(a) RF on raw 100 PCs:          {acc_raw:.3f}  (chance {chance:.3f})")

# (b) Z ALONE (trivial features)
acc_Z = cross_val_score(
    RandomForestClassifier(400, n_jobs=-1, random_state=0),
    result["Z"], y, cv=cv).mean()
print(f"(b) RF on Z (19 trivial):       {acc_Z:.3f}")

# (c) Z + residualized PCs
acc_both = cross_val_score(
    RandomForestClassifier(400, n_jobs=-1, random_state=0),
    np.hstack([result["Z"], result["scores"][:, :20]]), y, cv=cv).mean()
print(f"(c) RF on Z + 20 residual PCs:  {acc_both:.3f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np, pandas as pd

y = np.asarray(meta["method"])
cv = StratifiedKFold(5, shuffle=True, random_state=0)

def rf_acc(cols):
    if isinstance(cols, str):
        cols = [cols]
    X = result["Z_columns"]
    idx = [X.index(c) for c in cols]
    return cross_val_score(
        RandomForestClassifier(400, n_jobs=-1, random_state=0),
        result["Z"][:, idx], y, cv=cv).mean()

# Leave-one-feature-out: how much does each feature contribute?
print("Leave-one-out RF accuracy on Z:")
for c in result["Z_columns"]:
    others = [x for x in result["Z_columns"] if x != c]
    acc = rf_acc(others)
    print(f"  drop {c:>22s}: {acc:.3f}")

# Single-feature accuracies
print("\nSingle-feature RF accuracy:")
for c in result["Z_columns"]:
    acc = rf_acc(c)
    if acc > 0.2:
        print(f"  {c:>22s}: {acc:.3f}")

In [ ]:
# RF on raw PCs, BUT residualize the PCs against Z first
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# refit raw PCs (should already have pca_raw from the sanity test)
Z = result["Z"]
reg = LinearRegression().fit(Z, pca_raw)
pca_raw_resid = pca_raw - reg.predict(Z)

acc_resid = cross_val_score(
    RandomForestClassifier(400, n_jobs=-1, random_state=0),
    pca_raw_resid, y, cv=cv).mean()
print(f"RF on raw PCs residualized against Z: {acc_resid:.3f}")

In [ ]:
# 1) Find which Z columns carry the fingerprint
#    (use the leave-one-out loop above)

# 2) Check if raw signal survives residualization
#    (use the residualize-raw-PCs test above)

# 3) Look at what RF is using in the raw case
rf = RandomForestClassifier(800, n_jobs=-1, random_state=0)
rf.fit(pca_raw, y)
imp = rf.feature_importances_
print("Top raw PCs by importance:",
      np.argsort(imp)[::-1][:10], imp[np.argsort(imp)[::-1][:10]])

# 4) UMAP on raw PCs, colored by method — see if clusters exist
# pip install umap-learn
import umap
emb = umap.UMAP(n_neighbors=15, min_dist=0.1,
                random_state=0).fit_transform(pca_raw[:, :50])
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 6))
for m in np.unique(y):
    plt.scatter(emb[y == m, 0], emb[y == m, 1], s=8, alpha=0.6, label=m)
plt.legend(fontsize=7, markerscale=2)
plt.title("UMAP on raw 50 PCs")
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import numpy as np

rf = RandomForestClassifier(800, n_jobs=-1, random_state=0).fit(pca_raw_resid, y)
imp = permutation_importance(rf, pca_raw_resid, y, n_repeats=10,
                             n_jobs=-1, random_state=0)
order = np.argsort(imp.importances_mean)[::-1]

print("Top 15 PCs (after residualization) by permutation importance:")
for i in order[:15]:
    print(f"  PC{i:>3d}  imp={imp.importances_mean[i]:+.4f}  "
          f"std={imp.importances_std[i]:.4f}")

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

yhat = cross_val_predict(
    RandomForestClassifier(800, n_jobs=-1, random_state=0),
    pca_raw_resid, y, cv=5)
print(classification_report(y, yhat, digits=3))
print(confusion_matrix(y, yhat, labels=np.unique(y)))

In [ ]:
import umap
import matplotlib.pyplot as plt

emb = umap.UMAP(n_neighbors=15, min_dist=0.1,
                n_components=2, random_state=0).fit_transform(pca_raw_resid[:, :50])

fig, ax = plt.subplots(figsize=(8, 7))
for m in np.unique(y):
    ax.scatter(emb[y == m, 0], emb[y == m, 1],
               s=6, alpha=0.6, label=m)
ax.legend(fontsize=8, markerscale=3, loc="best")
ax.set_title("UMAP on residualized PCs (dynamics only)")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

acc_lin = cross_val_score(
    LogisticRegression(max_iter=5000),
    pca_raw_resid[:, :100], y, cv=5).mean()
print(f"LogReg on residualized PCs: {acc_lin:.3f}   "
      f"(RF was 0.507, chance 0.125)")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

# Proper CV: refit everything per fold
X_raw = np.stack([resample_sequence(s) for s in meta["steps"]])

class ResidualizeAgainstZ:
    def __init__(self, Z): self.Z = Z
    def fit(self, X, y=None):
        self.scaler = StandardScaler().fit(self.Z)
        Zs = self.scaler.transform(self.Z)
        self.reg = LinearRegression().fit(Zs, X)
        return self
    def transform(self, X):
        Zs = self.scaler.transform(self.Z)
        return X - self.reg.predict(Zs)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=100, random_state=0)),
    ("resid", ResidualizeAgainstZ(result["Z"])),
    ("rf", RandomForestClassifier(800, n_jobs=-1, random_state=0)),
])
# note: this pipe leaks Z from outside CV; for a strict number, use a
# custom CV loop. For a quick sanity check, it's fine.
acc = cross_val_score(pipe, X_raw, y, cv=5).mean()
print(f"CV with in-fold residualization: {acc:.3f}")

In [ ]:
print("X shape:", pca_raw_resid.shape)
print("y shape:", y.shape)
print("meta shape:", meta.shape)

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

class ResidualizeAgainstZ(BaseEstimator, TransformerMixin):
    """Residualize X against fixed external Z (fit inside fold)."""
    def __init__(self, Z, ridge_alpha=1.0):
        self.Z = Z
        self.ridge_alpha = ridge_alpha
    def fit(self, X, y=None):
        from sklearn.linear_model import Ridge
        self.z_scaler_ = StandardScaler().fit(self.Z)
        Zs = self.z_scaler_.transform(self.Z)
        self.reg_ = Ridge(alpha=self.ridge_alpha).fit(Zs, X)
        return self
    def transform(self, X):
        Zs = self.z_scaler_.transform(self.Z)
        return X - self.reg_.predict(Zs)

X_raw = np.stack([resample_sequence(s) for s in meta["steps"]])
y = np.asarray(meta["method"])
cv = StratifiedKFold(5, shuffle=True, random_state=0)

pipe_rf = Pipeline([
    ("scale",  StandardScaler()),
    ("pca",    PCA(n_components=100, random_state=0)),
    ("resid",  ResidualizeAgainstZ(result["Z"])),
    ("clf",    RandomForestClassifier(800, n_jobs=-1, random_state=0)),
])
pipe_lr = Pipeline([
    ("scale",  StandardScaler()),
    ("pca",    PCA(n_components=100, random_state=0)),
    ("resid",  ResidualizeAgainstZ(result["Z"])),
    ("clf",    LogisticRegression(max_iter=5000)),
])

print("RF   :", cross_val_score(pipe_rf, X_raw, y, cv=cv).mean())
print("LogR :", cross_val_score(pipe_lr, X_raw, y, cv=cv).mean())